# CV Coach Fine-Tuning (v2) - Kapsamlı Eğitim

Bu notebook, CV Coach modelini **800+ örneklik** (400 Türkçe + 400 İngilizce, çok turlu sohbet ve skor odaklı sorular içeren) kapsamlı bir veri setiyle eğitir.

## Adımlar
1. **Hücre 1**: Kütüphaneleri kur (5-7 dk)
2. **Hücre 2**: Repo'yu çek (yeni veri üreticisi + 800 örneklik veri seti)
3. **Hücre 3**: Eğitimi başlat (~15-20 dk, T4 GPU)
4. **Hücre 4**: Eğitim sonrası hızlı test (TR + EN + skor sorusu)
5. **Hücre 5**: Zip'i indir → Mac'te `setup_ollama.sh` ile yükle

## Hücre 1 - Kütüphaneler

Çalıştırıp tamamlanmasını bekleyin (altta [OK] yazısı çıkmalı).

In [ ]:
!pip install -q transformers accelerate peft datasets bitsandbytes scipy sentencepiece
!pip install -q -U bitsandbytes
print("[OK] Kutuphaneler hazir")

## Hücre 2 - Repo + Veri Seti

GitHub'dan son haliyle çekilir; `cv_coach_dataset.json` (800 örnek) hazır gelir. Veri setinin doğru geldiğini doğrular.

In [ ]:
import json, os

os.chdir('/content')
!rm -rf /content/cvmatcher
!git clone --depth 1 https://github.com/ruveydagundogan/cvmatcher.git /content/cvmatcher
os.chdir('/content/cvmatcher')
print('[OK] Repo cekildi:', os.getcwd())

with open('backend/finetune/data/cv_coach_dataset.json', encoding='utf-8') as f:
    data = json.load(f)
tr = sum(1 for x in data if x['system'].startswith('Sen CV Koçusun'))
multi = sum(1 for x in data if 'history' in x)
score = sum(1 for x in data if any(k in x['input'] for k in ['skor','score','eşleşme','match','döküm','breakdown']))
print(f'[OK] Veri seti: {len(data)} ornek ({tr} TR / {len(data)-tr} EN), cok turlu: {multi}, skor sorusu: {score}')

## Hücre 3 - Eğitim

Qwen2.5-1.5B-Instruct + LoRA (r=16). Eğitim sırasında **loss değerlerini ve step/epoch ilerlemesini** izleyin. 20-30 dk sürebilir.

In [ ]:
import torch
print('[OK] torch', torch.__version__, '| CUDA:', torch.cuda.is_available())
print('[OK] GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')

!python backend/finetune/train_lora.py \
  --mode cv-coach \
  --base-model Qwen/Qwen2.5-1.5B-Instruct \
  --data backend/finetune/data/cv_coach_dataset.json \
  --output-dir backend/finetune/adapters-clean/cv-coach-v1 \
  --epochs 3 \
  --batch-size 4 \
  --max-length 512 \
  --quantize \
  --lora-r 16 \
  --lora-alpha 32
print('\n[OK] Egitim tamamlandi (veya hata yukarida)')

## Hücre 4 - Eğitim Sonrası Test

Eğitilen adapter'ı Colab'da doğrudan test eder: Türkçe skor sorusu, İngilizce skor sorusu ve genel soru. Doğru skorlar (örn. 25/100) aynen tekrarlanmalı.

In [ ]:
!pip install -q -U torchao
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

base = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-1.5B-Instruct",
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
model = PeftModel.from_pretrained(base, "backend/finetune/adapters-clean/cv-coach-v1")
model.eval()
tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")

SYSTEM = (
    "You are the CV Coach, an expert career assistant. Be concrete, practical and encouraging. "
    "ALWAYS answer in the same language the user writes in. Quote the exact scores as written "
    "in the context (for example \"25/100\"). Never invent scores.\n\n"
    "The AI match result: Overall score: 25/100. Skill match: 0/100, Experience: 50/100, "
    "Education: 50/100. Matched skills: Python, Docker. Missing skills: FastAPI, Redis, Kafka, Kubernetes.\n\n"
    "CV context: Title: Backend Developer. Skills: Python, Django, PostgreSQL, Docker.\n\n"
    "Job description: Title: Backend Engineer. Required skills: FastAPI, Redis, Kafka, Kubernetes."
)

def ask(user_msg):
    msgs = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": user_msg},
    ]
    prompt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    out = model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=True,
        temperature=0.6,
        top_p=0.9,
        repetition_penalty=1.3,
    )
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

tests = [
    ("Toplam eşleşme skorum kaç?", "25/100"),
    ("What is my overall match score?", "25/100"),
    ("Deneyim skorumu nasıl iyileştiririm?", "50/100"),
]
for q, expect in tests:
    print('=' * 70)
    print('SORU :', q)
    a = ask(q)
    print('CEVAP:', a)
    print('SKOR :', 'OK' if expect in a else f'EXPECTED {expect}')

## Hücre 5 - Zip'i İndir

`cvmatcher-lora.zip` dosyası indirilir. Sonra Mac'te:

```bash
bash backend/finetune/setup_ollama.sh ~/Downloads/cvmatcher-lora.zip
ollama stop cv-coach
ollama run cv-coach
```

In [ ]:
!pip install -q -U torchao
import shutil, os

out_dir = 'backend/finetune/adapters-clean/cv-coach-v1'
for ckpt in [d for d in os.listdir(out_dir) if d.startswith('checkpoint')]:
    shutil.rmtree(os.path.join(out_dir, ckpt))
print('[OK] Checkpoint temizlendi')

!zip -r /content/cvmatcher-lora.zip backend/finetune/adapters-clean/cv-coach-v1
from google.colab import files
files.download('/content/cvmatcher-lora.zip')
print('[OK] cvmatcher-lora.zip indirildi')